# Day 2 Sample Notebook: Concept Sets and Data Quality

**OHDSI / OMOP Train-the-Trainer, ALS Therapy Development Institute**

Companion to the Day 2 module. You will build a concept set using descendants, then run the three
Kahn data quality categories (conformance, completeness, plausibility) against a synthetic ALS CDM.

> **Site specific.** This notebook builds a synthetic CDM in memory so it runs in Colab with no credentials and no PHI. To use real data, replace the SQLite connection with your site's governed CDM connection (Databricks, Snowflake, Postgres, BigQuery, and so on). The SQL logic is the same; only the connection changes. Not everyone uses Databricks.

### Objectives
1. Build a concept set with descendants and validate it with SQL.
2. Run a conformance check (are concepts valid and mapped?).
3. Run a completeness check (are expected values present?).
4. Run a plausibility check (are values believable?).

## Step 0: Build the synthetic CDM

In [ ]:
import sqlite3, numpy as np, pandas as pd
np.random.seed(7)

def build_als_cdm(n=250):
    """Build a tiny SYNTHETIC ALS-flavored OMOP CDM in memory. No real data, no PHI."""
    con = sqlite3.connect(":memory:"); cur = con.cursor()
    cur.executescript("""
    CREATE TABLE concept(concept_id INT, concept_name TEXT, domain_id TEXT, vocabulary_id TEXT,
        concept_class_id TEXT, standard_concept TEXT, concept_code TEXT);
    CREATE TABLE concept_ancestor(ancestor_concept_id INT, descendant_concept_id INT, min_levels_of_separation INT);
    CREATE TABLE person(person_id INT, gender_concept_id INT, year_of_birth INT);
    CREATE TABLE observation_period(person_id INT, observation_period_start_date TEXT, observation_period_end_date TEXT);
    CREATE TABLE condition_occurrence(person_id INT, condition_concept_id INT, condition_start_date TEXT);
    CREATE TABLE drug_exposure(person_id INT, drug_concept_id INT, drug_exposure_start_date TEXT);
    CREATE TABLE measurement(person_id INT, measurement_concept_id INT, measurement_date TEXT, value_as_number REAL);
    CREATE TABLE procedure_occurrence(person_id INT, procedure_concept_id INT, procedure_date TEXT);
    """)
    concepts=[
     (35748069,"Amyotrophic lateral sclerosis","Condition","SNOMED","Clinical Finding","S","86044005"),
     (4134454,"Bulbar onset","Observation","SNOMED","Clinical Finding","S","230258005"),
     (1314865,"riluzole","Drug","RxNorm","Ingredient","S","9468"),
     (19077572,"riluzole 50 MG Oral Tablet","Drug","RxNorm","Clinical Drug","S","316158"),
     (1326727,"edaravone","Drug","RxNorm","Ingredient","S","1546020"),
     (40234834,"edaravone 30 MG/100mL","Drug","RxNorm","Clinical Drug","S","1546022"),
     (9990001,"sodium phenylbutyrate / taurursodiol","Drug","RxNorm","Ingredient","S","2630918"),
     (9990002,"tofersen","Drug","RxNorm","Ingredient","S","2629000"),
     (1304643,"baclofen","Drug","RxNorm","Ingredient","S","1292"),
     (4052536,"Gastrostomy","Procedure","SNOMED","Procedure","S","36245001"),
     (3024171,"Forced vital capacity","Measurement","LOINC","Clinical Obs","S","19868-9"),
     (8532,"FEMALE","Gender","Gender","Gender","S","F"),(8507,"MALE","Gender","Gender","Gender","S","M"),
     (0,"No matching concept","Metadata","None","Undefined",None,"0")]
    cur.executemany("INSERT INTO concept VALUES(?,?,?,?,?,?,?)",concepts)
    cur.executemany("INSERT INTO concept_ancestor VALUES(?,?,?)",[
     (1314865,19077572,1),(1314865,1314865,0),(1326727,40234834,1),(1326727,1326727,0),
     (9990001,9990001,0),(9990002,9990002,0),(1304643,1304643,0)])
    persons=[];obsp=[];cond=[];drug=[];meas=[];proc=[]
    for pid in range(1,n+1):
        sex=int(np.random.choice([8532,8507])); yob=int(np.random.normal(1958,11))
        persons.append((pid,sex,yob))
        dx_year=np.random.randint(2018,2024); dx=f"{dx_year}-{np.random.randint(1,13):02d}-15"
        cond.append((pid,35748069,dx))
        bulbar=np.random.rand()<0.3
        if bulbar: cond.append((pid,4134454,dx))
        prior=int(np.random.choice([400,500,800,200,100],p=[.4,.2,.2,.1,.1]))
        start=pd.Timestamp(dx)-pd.Timedelta(days=prior)
        end=pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(365,1500)))
        obsp.append((pid,start.date().isoformat(),end.date().isoformat()))
        base_fvc=float(np.clip(np.random.normal(85-(10 if bulbar else 0),12),30,110))
        r=np.random.rand()
        if r<0.08: pass                                  # missing (completeness)
        elif r<0.11: meas.append((pid,3024171,dx,float(np.random.choice([0,250.0]))))  # implausible (plausibility)
        else: meas.append((pid,3024171,dx,round(base_fvc,1)))
        idx=pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(0,40)))
        if np.random.rand()<0.85:
            drug.append((pid,19077572,idx.date().isoformat()))
            if np.random.rand()<0.45:
                drug.append((pid,40234834,(idx+pd.Timedelta(days=int(np.random.randint(60,300)))).date().isoformat()))
        else:
            drug.append((pid,40234834,idx.date().isoformat()))
        if np.random.rand()<0.15:
            drug.append((pid,9990001,(idx+pd.Timedelta(days=int(np.random.randint(100,500)))).date().isoformat()))
        if bulbar and np.random.rand()<0.4: drug.append((pid,1304643,idx.date().isoformat()))
        if np.random.rand()<0.05: drug.append((pid,0,idx.date().isoformat()))   # unmapped (conformance)
        age=dx_year-yob
        logit=-2.0+0.04*(age-60)+1.1*bulbar-0.03*(base_fvc-80)
        if np.random.rand()<1/(1+np.exp(-logit)):
            proc.append((pid,4052536,(pd.Timestamp(dx)+pd.Timedelta(days=int(np.random.randint(60,360)))).date().isoformat()))
    cur.executemany("INSERT INTO person VALUES(?,?,?)",persons)
    cur.executemany("INSERT INTO observation_period VALUES(?,?,?)",obsp)
    cur.executemany("INSERT INTO condition_occurrence VALUES(?,?,?)",cond)
    cur.executemany("INSERT INTO drug_exposure VALUES(?,?,?)",drug)
    cur.executemany("INSERT INTO measurement VALUES(?,?,?,?)",meas)
    cur.executemany("INSERT INTO procedure_occurrence VALUES(?,?,?)",proc)
    con.commit(); return con

con = build_als_cdm()
def q(sql): return pd.read_sql_query(sql, con)
print("Synthetic ALS CDM ready:", q("SELECT COUNT(*) n FROM person").n[0], "persons")

## Step 1: A concept set with descendants
A concept set for the riluzole ingredient should capture every product below it. The `concept_ancestor`
table makes this one join instead of listing each product.

In [ ]:
q("""
SELECT ing.concept_name AS ingredient, d.concept_name AS product_in_set
FROM concept_ancestor ca
JOIN concept ing ON ing.concept_id = ca.ancestor_concept_id
JOIN concept d   ON d.concept_id   = ca.descendant_concept_id
WHERE ca.ancestor_concept_id = 1314865
""")

## Step 2: Conformance
Do all drug records point at a valid, standard concept? Records on `concept_id = 0` did not map.

In [ ]:
q("""
SELECT CASE WHEN drug_concept_id = 0 THEN 'unmapped (concept_id 0)' ELSE 'mapped' END AS status,
       COUNT(*) AS rows
FROM drug_exposure GROUP BY status
""")

## Step 3: Completeness
Every person in an ALS cohort should have a baseline FVC measurement. How many are missing one?

In [ ]:
q("""
SELECT COUNT(*) AS als_persons,
       SUM(CASE WHEN m.person_id IS NULL THEN 1 ELSE 0 END) AS missing_fvc
FROM person p
LEFT JOIN measurement m
  ON m.person_id = p.person_id AND m.measurement_concept_id = 3024171
""")

## Step 4: Plausibility
FVC percent predicted should fall in a believable range (roughly 1 to 120). Flag values outside it.

In [ ]:
q("""
SELECT value_as_number,
       CASE WHEN value_as_number <= 0 OR value_as_number > 120 THEN 'implausible' ELSE 'ok' END AS flag
FROM measurement
WHERE measurement_concept_id = 3024171 AND (value_as_number <= 0 OR value_as_number > 120)
""")

## Your turn
1. Build the descendant set for **edaravone** (ingredient 1326727) and count exposed persons.
2. Completeness: what fraction of ALS persons are missing an FVC? Express it as a percentage.
3. Plausibility: write a check for birth years that are implausible (for example before 1900 or in the future).

<details><summary>Answer key</summary>

```sql
-- 1
SELECT COUNT(DISTINCT de.person_id) FROM drug_exposure de
JOIN concept_ancestor ca ON ca.descendant_concept_id=de.drug_concept_id
WHERE ca.ancestor_concept_id=1326727;
-- 2
SELECT ROUND(100.0*SUM(CASE WHEN m.person_id IS NULL THEN 1 ELSE 0 END)/COUNT(*),1)
FROM person p LEFT JOIN measurement m ON m.person_id=p.person_id AND m.measurement_concept_id=3024171;
-- 3
SELECT person_id, year_of_birth FROM person WHERE year_of_birth < 1900 OR year_of_birth > 2025;
```
Discussion: which of these three failures would actually block an analysis, and which are cosmetic?
</details>